## Indexing/Retrieval and Generation

The image below illustrates the overall components of a naive RAG Pipeline, which includes 3 broad stages _Indexing_, _Retrieval_ and _Generation_. In this notebook, we'll cover all 3 stages and build the _naive RAG pipeline_.

![RAG Pipeline](../images/rag_pipeline.png)

The first aspect of indexing is the process of _loading_. Here we load all our documents and put them into what we call a _Retriever_. The goal of the _Retriever_ is - given an inpu


## Modules Setup
You will need to install the following Python modules to begin with
```bash
$> uv add langchain_community tiktoken langchain-openai langchainhub chromadb langchain
```

In [66]:
from dotenv import load_dotenv
from rich.console import Console

import langchain

import langchainhub as hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

print(f"Using Langchain {langchain.__version__}")

Using Langchain 1.2.14


In [67]:
load_dotenv(override=True)
console = Console()

## Step 01 - Indexing

For this example, we'll be indexing Meta's **[Llama 3 Technical Report](https://arxiv.org/abs/2407.21783)** (arXiv:2407.21783, July 2024) — a 90+ page paper that describes the architecture, training methodology, safety work, and benchmark results for the entire Llama 3 family of models. It's a good choice for testing a RAG pipeline because:

- It is a **post-2023 document** — not in GPT-3.5-turbo's training data, so the model cannot fall back on memorised knowledge.
- It is **long and multi-sectional** — architecture details, pre-training, post-training, safety, and benchmarks each live in separate sections, making cross-section retrieval genuinely hard.
- It is **dense with specific facts** — layer counts, attention heads, benchmark scores, and training recipes that require precise retrieval to answer correctly.

Indexing follows 3 steps:

1. **Loading** — we use LangChain's `PyPDFLoader` to parse the local PDF file page by page into a list of `Document` objects.
2. **Splitting** — we use `RecursiveCharacterTextSplitter` to break each page into overlapping chunks of ~1000 characters. Overlapping ensures that sentences spanning page or chunk boundaries are not silently cut off.
3. **Embedding & Storing** — each chunk is embedded using OpenAI embeddings and stored in a **Chroma** vector store, which will serve as our retriever.


In [68]:
# Step 1 - Load the PDF (must be in the same folder as this notebook)
loader = PyPDFLoader("Llama3TechnicalReport.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages from PDF")

# Step 2 - Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks")

# Step 3 - Embed and store
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

Loaded 92 pages from PDF
Split into 462 chunks


In [69]:
prompt = hub.Client().pull("rlm/rag-prompt")
console.print(prompt)

C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_6268\890926960.py:1: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = hub.Client().pull("rlm/rag-prompt")


{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": 
{"messages": [{"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": 
"constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": 
"constructor", "kwargs": {"template": "You are an assistant for question-answering tasks. Use the following pieces 
of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three 
sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:", 
"input_variables": ["question", "context"], "template_format": "f-string"}}}}], "input_variables": ["question", 
"context"]}}

In [70]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces 
of retrieved context to answer the question. If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
\nQuestion: {question} \nContext: {context}
\nAnswer:"""
)

In [71]:
# create my LLM
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

In [72]:
# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Chain
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [73]:
# Simple, focused question - naive RAG handles this fine
rag_chain.invoke("What context window length does Llama 3 support?")

'Llama 3 supports a context window length of 128K tokens during the final pre-training stage. The balance between short and long-context capabilities is carefully tuned during finetuning. Incorporating long-context data is essential for maintaining long-context capabilities in the model.'

**How does this all work?**

* LangChain provides a lot of classes, such as retrievers, ChatPromptTemplate, your LLMs, parsers (such as StrOutputParser). 
* Each of these objects are derived from a `Runnable` class - this is the secret sauce! 
* The `Runnable` class provides an `invoke()` method, so all `Runnable`s can be `invoke()`-ed! 
* Further, all `Runnables` can be chained together using the `|` operator, giving you a chain of execution, which is also a `Runnable`! 

> **Wait a minute!** 💡
> 
> But `{"context": retriever | RunnableLambda(format_docs),"question": RunnablePassthrough(),}` is a Python dict and _NOT_ a `Runnable`!<br/>
So how does this become a runnable??<br/>
`{"context": retriever | RunnableLambda(format_docs),"question": RunnablePassthrough(),}` | prompt
<br/><br/>

Great observation 👀! The trick is Python's operator protocol and LangChain's __ror__ implementation. Here is a _technical_ explanation - you don't have to fully understand it though!

When Python evaluates `dict | prompt`, it:
* First tries `dict.__or__(prompt)` — a plain dict doesn't know about `Runnable`s, so returns `NotImplemented`
* Falls back to `prompt.__ror__(dict)` — LangChain's `Runnable` base class _does implement_ __ror__, and it intercepts the dict here

Simple explanation: `dict | prompt` FAILS, but `prompt | dict` works and returns a `Runnable` 🤔🤩

**And what is `RunnableLambda(...)`?** 🤔

Often you want to chain your own plan Python functions into a chain of `Runnable`s. The `RunnableLambda` converts a regular Python function into a Runnable, which means you can now chain easily! 🤩

**And `RunnablePassthrough()`?** 🤔

In a chain, you will usually have some variables. In our specific example, we have variables such as `context` and `question` in our prompt. These variables need to have values assigned to them before the LLM can process them.

Here is the magic:
```python
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
```
This Python `dict` is providing the value for `context` by chaining retriever with our RunnableLambda. 

```python
    "context": retriever | RunnableLambda(format_docs)
```

The above line essentially means - take whatever the retriever gets and feed it to format_docs to get a set of paras separated by 2 blank lines (which is what `format_docs()` does!). The output you get _is_ the value of your `context` variable.

```python
    "question": RunnablePassthrough(),
```

The above line says, value of "question" is whatever _input_ you get from `rag_chain.invoke("What is Task Decomposition?")` call. So `"question" = "What is Task Decomposition?"` (i.e. whatever is passed from outer call passes through literally!) 🤩

In [74]:
# Compound query spanning two distant sections - watch naive RAG struggle
rag_chain.invoke(
    """What safety evaluation benchmarks does Llama 3 score highest on, and how does
    the RLHF post-training methodology specifically differ from the supervised fine-tuning
    approach used for coding capabilities?"""
)

'Llama 3 scores highest on safety evaluation benchmarks by focusing on generating content responsibly while maximizing helpful information. The RLHF post-training methodology differs from supervised fine-tuning by emphasizing safety characteristics in the pre-training stage. The safety work for Llama 3 begins early in the training process to ensure safe and responsible content generation.'

In [75]:
# Another cross-section compound query - naive RAG retrieves the wrong chunks
rag_chain.invoke(
    """How many attention heads does the 405B parameter Llama 3 model have, and how does
    that architectural choice impact its multilingual benchmark scores compared to the 8B variant?"""
)

'The 405B parameter Llama 3 model has 128 attention heads. This architectural choice likely contributes to its improved multilingual benchmark scores compared to the 8B variant.'

Do you see a problem with the responses to the above queries? Probably not, unless you read the PDF thoroughly! While  the model is generating plausible-sounding but unreliable answers, not genuinely answering from retrieved context. 

Look closely:
* **The benchmark query**: No specific benchmark names or scores cited — just vague generalities about "cybersecurity" and "chemical/biological weapons." A real answer from the paper would name exact test sets with numbers.
* **The attention heads query**: 128 attention heads → outperforms on MGSM" — the causal link between those two facts is fabricated. The paper never makes that argument. The model stitched together two unrelated retrieved fragments and invented a narrative connecting them.

In [76]:
# What did the retriever actually fetch for this compound query?
question = """What safety evaluation benchmarks does Llama 3 score highest on, and how does
    the RLHF post-training methodology specifically differ from the supervised fine-tuning
    approach used for coding capabilities?"""

for i, doc in enumerate(retriever.invoke(question)):
    print(f"--- Chunk {i+1} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:400])
    print()

--- Chunk 1 (page 39) ---
best-performing openly available model.
Limitations. All human evaluation results underwent a thorough data quality assurance process. However,
since it is challenging to define objective criteria for evaluating model responses, human evaluations can still
be influenced by personal biases, backgrounds, and preferences of human annotators, which may lead to
inconsistent or unreliable results.
5.4 S

--- Chunk 2 (page 39) ---
best-performing openly available model.
Limitations. All human evaluation results underwent a thorough data quality assurance process. However,
since it is challenging to define objective criteria for evaluating model responses, human evaluations can still
be influenced by personal biases, backgrounds, and preferences of human annotators, which may lead to
inconsistent or unreliable results.
5.4 S

--- Chunk 3 (page 27) ---
language model, (2) the post-trained language model, and(3) the safety characteristics of Llama 3. We present
the resu

You'll see that the 4 retrieved chunks address one half of the query at best — and the LLM filled the gap with confident-sounding fabrication. That's the real failure of naive RAG, and it's a much more honest and compelling segue into query translation than a broken answer.

These are not complete responses from the LLM. Why is that happening? We'll answer that in the next notebook in the series.